# Bank Customer Churn Prediction using a Neural Network
Dataset: `Churn_Modelling.csv` (Kaggle: barelydedicated/bank-customer-churn-modeling)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

np.random.seed(42)
tf.random.set_seed(42)

## 1. Read the dataset

In [ ]:
df = pd.read_csv('Churn_Modelling.csv')
df.head()

In [ ]:
print(df.shape)
print(df.isnull().sum())
print(df['Exited'].value_counts(normalize=True))  # ~80/20 imbalance
sns.countplot(x='Exited', data=df)
plt.show()

## 2. Features / target and train-test split
`RowNumber`, `CustomerId` and `Surname` are identifiers with no predictive value, so they are dropped. `Geography` and `Gender` are one-hot encoded.

In [ ]:
X = df.drop(columns=['RowNumber', 'CustomerId', 'Surname', 'Exited'])
y = df['Exited']

X = pd.get_dummies(X, columns=['Geography', 'Gender'], drop_first=True)
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## 3. Normalize the train and test data
The scaler is fit on the training set only, to avoid data leakage.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## 4a. Baseline model

In [ ]:
def build_baseline(n_features):
    model = models.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(6, activation='relu'),
        layers.Dense(6, activation='relu'),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

baseline = build_baseline(X_train.shape[1])
baseline.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=0)

y_pred_base = (baseline.predict(X_test, verbose=0) > 0.5).astype(int).ravel()
print('Baseline accuracy:', accuracy_score(y_test, y_pred_base))
print(confusion_matrix(y_test, y_pred_base))

## 4b. Improvements
- Wider / deeper network
- Dropout + BatchNormalization (regularisation, stable training)
- Class weights (handles the 80/20 imbalance, improves recall on churners)
- EarlyStopping + ReduceLROnPlateau (avoids overfitting)
- Decision threshold tuned on validation data

In [ ]:
def build_improved(n_features):
    model = models.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['accuracy'])
    return model

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))
print(class_weight)

# validation split from training data (used for early stopping and threshold tuning)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

In [ ]:
model = build_improved(X_train.shape[1])
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=200, batch_size=32,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5),
    ],
    verbose=0,
)

plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')
plt.legend(); plt.title('Training curves'); plt.show()

In [ ]:
# pick the threshold that maximises validation accuracy
val_probs = model.predict(X_val, verbose=0).ravel()
thresholds = np.arange(0.3, 0.8, 0.01)
best_t = max(thresholds, key=lambda t: accuracy_score(y_val, (val_probs > t).astype(int)))
print(f'Best threshold (validation): {best_t:.2f}')

## 5. Accuracy score and confusion matrix (test set)

In [ ]:
y_pred = (model.predict(X_test, verbose=0).ravel() > best_t).astype(int)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f'Accuracy score: {acc:.4f}')
print('Confusion matrix:\n', cm)
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Left']))

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=['Stayed', 'Left'], yticklabels=['Stayed', 'Left'])
plt.title('Confusion matrix')
plt.xlabel('Predicted label')
plt.ylabel('Actual label')
plt.show()